# YAICON Qwen TTS GPU trial

Private inference benchmark for an explicitly labeled AI persona simulation.
Both pinned Qwen3-TTS Base models read the same complete English answer, using
the same reference and transcript as the Mac baseline. One CUDA device is used.
The first request includes loading and prompt construction. The next two use
the cached reference. Output contains timing, provenance, and example WAVs.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import base64
import hashlib
import io
import os
import subprocess
import sys
import tarfile
import tempfile
import torch

assert torch.cuda.is_available(), "This experiment requires a Kaggle GPU."
print("GPU:", torch.cuda.get_device_name(0), "CUDA:", torch.version.cuda, flush=True)
work = Path(tempfile.mkdtemp(prefix="yaicon-voice-"))
out = Path("/kaggle/working")
model_cache = work / "model-cache"
os.environ.update(HF_HUB_DISABLE_TELEMETRY="1", HF_HUB_DISABLE_XET="1",
                  HF_HUB_DOWNLOAD_TIMEOUT="90", TOKENIZERS_PARALLELISM="false",
                  PIP_DISABLE_PIP_VERSION_CHECK="1", USE_TF="0", HF_HUB_DISABLE_PROGRESS_BARS="1")


In [ ]:
# Attach a private Kaggle input containing the experiment assets, then set this path.
input_path = Path("/kaggle/input/CHANGE-ME")
blob = (input_path / "gpu-voice-trial.tar.gz").read_bytes()
bundle = work / "bundle"
bundle.mkdir(mode=0o700)
with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as archive:
    for member in archive.getmembers():
        relative_path = Path(member.name)
        assert member.isfile() and not relative_path.is_absolute() and ".." not in relative_path.parts
        path = bundle / relative_path
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(archive.extractfile(member).read())
subprocess.run([sys.executable, "gpu_voice_trial.py", "check"], cwd=bundle, check=True)


In [ ]:
python = sys.executable
constraints = work / "constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch", "torchvision", "torchaudio")))
print("Installing the voice dependencies in the disposable Kaggle session, preserving its CUDA packages.", flush=True)
subprocess.run([python, "-m", "pip", "install", "--break-system-packages", "-q",
                "-c", str(constraints), "-r", str(bundle / "services/voice/requirements.txt")],
               timeout=360, check=True)


In [ ]:
runner = bundle / "kaggle_runner.py"
runner.write_text('from pathlib import Path\nimport hashlib\nimport io\nimport json\nimport statistics\nimport sys\nimport wave\nfrom time import perf_counter\n\nimport torch\nfrom huggingface_hub import snapshot_download\nfrom local_voice import load_full_text_model\nfrom services.voice.engine import VoiceEngine, VoiceSettings\n\ntag, revision, cache, output = sys.argv[1:]\nroot = Path.cwd()\noutput = Path(output)\nfixture = json.loads((root / "benchmark.json").read_text())\nmodel_id = f"Qwen/Qwen3-TTS-12Hz-{tag}-Base"\nassert torch.cuda.is_available()\ndtype = "float32"\nprint(f"{tag}: downloading pinned weights; dtype={dtype}", flush=True)\nstarted = perf_counter()\nsnapshot_download(repo_id=model_id, revision=revision, cache_dir=cache,\n                  allow_patterns=["*.safetensors", "*.json", "*.txt"])\ndownload_seconds = perf_counter() - started\n\nsettings = VoiceSettings(model_id=model_id, revision=revision, cache_dir=Path(cache), device="cuda:0",\n                         dtype=dtype, attention="sdpa", reference_wav=root / "reference.wav",\n                         reference_text=(root / "reference.txt").read_text().strip(),\n                         max_input_chars=3000, max_new_tokens=1000, max_audio_seconds=80)\nengine = VoiceEngine(settings, model_loader=load_full_text_model)\nresult = {\n    "label": "AI persona simulation, experimental reference voice", "model": model_id, "revision": revision,\n    "gpu": torch.cuda.get_device_name(0), "visible_gpus": torch.cuda.device_count(),\n    "gpu_vram_bytes": torch.cuda.get_device_properties(0).total_memory,\n    "torch": torch.__version__, "cuda_version": torch.version.cuda, "dtype": dtype,\n    "download_seconds": round(download_seconds, 3),\n    "reference_sha256": hashlib.sha256((root / "reference.wav").read_bytes()).hexdigest(),\n    "answer_sha256": hashlib.sha256(fixture["text"].encode()).hexdigest(),\n    "mac_warm_synthesis_ms": fixture["mac_warm_synthesis_ms"], "runs": [],\n}\ntorch.cuda.reset_peak_memory_stats()\nfor i in range(3):\n    torch.cuda.synchronize()\n    start = perf_counter()\n    wav_data = engine.synthesize(fixture["text"], fixture["language"])\n    torch.cuda.synchronize()\n    elapsed = perf_counter() - start\n    name = f"experimental-ai-voice-{tag}-{i+1}.wav"\n    with wave.open(io.BytesIO(wav_data), "rb") as audio:\n        assert audio.getnframes() > 0 and audio.getnchannels() == 1 and audio.getsampwidth() == 2\n        duration = audio.getnframes() / audio.getframerate()\n    (output / name).write_bytes(wav_data)\n    result["runs"].append({**engine.status()["last_generation"], "wav": name,\n                           "sha256": hashlib.sha256(wav_data).hexdigest(),\n                           "wall_seconds": round(elapsed, 3), "duration_seconds": round(duration, 3),\n                           "peak_allocated_bytes": torch.cuda.max_memory_allocated()})\n    print(f"{tag}: attempt {i+1}, {elapsed:.3f}s for {duration:.3f}s of audio", flush=True)\n    (output / f"result-{tag}.json").write_text(json.dumps(result, indent=2) + "\\n")\nresult["median_warm_seconds"] = statistics.median(run["wall_seconds"] for run in result["runs"][1:])\n(output / f"result-{tag}.json").write_text(json.dumps(result, indent=2) + "\\n")\n')
models = [("0.6B", "5d83992436eae1d760afd27aff78a71d676296fc"),
          ("1.7B", "fd4b254389122332181a7c3db7f27e918eec64e3")]
for tag, revision in models:
    subprocess.run([python, str(runner), tag, revision, str(model_cache), str(out)],
                   cwd=bundle, timeout=600, check=True)
